In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import cv2

def draw_circle_closest_to_center(image, circle_coordinates):
    """
    Draw the largest circle among the three closest to the center of the image.
    
    Args:
        image (numpy.ndarray): The original image.
        circle_coordinates (list of tuples): List of (x, y, r) coordinates of circles.
        
    Returns:
        numpy.ndarray: Image with the circle drawn.
    """
    center_x = image.shape[1] // 2
    center_y = image.shape[0] // 2
    
    circle_coordinates = sorted(circle_coordinates, key=lambda c: np.sqrt((c[0] - center_x)**2 + (c[1] - center_y)**2))
    closest_three_circles = circle_coordinates[:3]
    largest_circle = max(closest_three_circles, key=lambda c: c[2])
    
    if largest_circle is not None:
        (x, y, r) = largest_circle
        cv2.circle(image, (x, y), r, (0, 255, 0), 4)
        cv2.rectangle(image, (x - 5, y - 5), (x + 5, y + 5), (0, 128, 255), -1)
    
    return image, largest_circle

def mask_outside_circle(image, circle):
    """
    Mask the area outside the given circle and draw the circle on the image.
    
    Args:
        image (numpy.ndarray): The input image.
        circle (tuple): The (x, y, r) coordinates of the circle.
        
    Returns:
        numpy.ndarray: The masked image with the circle drawn.
    """
    mask = np.zeros(image.shape[:2], dtype="uint8")
    (x, y, r) = circle
    cv2.circle(mask, (x, y), r, 255, -1)
    
    masked_image = cv2.bitwise_and(image, image, mask=mask)
    cv2.circle(masked_image, (x, y), r, (0, 255, 0), 4)
    
    return masked_image, mask

def process_images_in_directory(directory_path):
    for filename in os.listdir(directory_path):
        if filename.endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tiff')):
            image_path = os.path.join(directory_path, filename)
            print(f"Processando {image_path}")
            
            image = cv2.imread(image_path)
            image1 = image.copy()
            output = image.copy()
            gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
            edges = cv2.Canny(gray, 100, 250)
            edges = cv2.GaussianBlur(edges, (25, 25), 2)
            edges2 = edges.copy()

            circles_original = cv2.HoughCircles(gray, cv2.HOUGH_GRADIENT, 1.2, 250, minRadius=90, maxRadius=220)
            circles_edges = cv2.HoughCircles(edges, cv2.HOUGH_GRADIENT, 1.2, 200, minRadius=90, maxRadius=300)

            if circles_original is not None:
                circles_original = np.round(circles_original[0, :]).astype("int")
            if circles_edges is not None:
                circles_edges = np.round(circles_edges[0, :]).astype("int")

            if circles_original is not None:
                for (x, y, r) in circles_original:
                    cv2.circle(output, (x, y), r, (0, 255, 0), 4)
                    cv2.rectangle(output, (x - 5, y - 5), (x + 5, y + 5), (0, 128, 255), -1)

            image_rgb = cv2.cvtColor(edges2, cv2.COLOR_BGR2RGB)

            if circles_edges is not None:
                for (x, y, r) in circles_edges:
                    cv2.circle(image_rgb, (x, y), r, (0, 255, 0), 4)
                    cv2.rectangle(image_rgb, (x - 5, y - 5), (x + 5, y + 5), (0, 128, 255), -1)
                    print("Circle detected at center (x={}, y={}), radius={}".format(x, y, r))

            if circles_edges is not None:
                print("Number of circles detected in edges image:", len(circles_edges))

            output_rgb = cv2.cvtColor(output, cv2.COLOR_BGR2RGB)
            edges_rgb = cv2.cvtColor(edges, cv2.COLOR_BGR2RGB)

            plt.figure(figsize=(20, 10))

            plt.subplot(1, 7, 1)
            plt.imshow(image_rgb)
            plt.title("Edges Circles")
            plt.axis('off')

            plt.subplot(1, 7, 2)
            plt.imshow(output_rgb)
            plt.title("Original Circles")
            plt.axis('off')

            if circles_edges is not None:
                closest_circle_edges_output, largest_circle_edges = draw_circle_closest_to_center(image.copy(), circles_edges.tolist())
                
                plt.subplot(1, 7, 3)
                plt.imshow(closest_circle_edges_output)
                plt.title("Closest Circle Edges")
                plt.axis('off')

                masked_edges, mask = mask_outside_circle(image.copy(), largest_circle_edges)
                masked_edges_rgb = cv2.cvtColor(masked_edges, cv2.COLOR_BGR2RGB)
                plt.subplot(1, 7, 4)
                plt.imshow(masked_edges_rgb)
                plt.title("Masked Closest Circle Edges")
                plt.axis('off')

                hsv_image = cv2.cvtColor(masked_edges_rgb, cv2.COLOR_RGB2HSV)
                hsv_image_masked = cv2.bitwise_and(hsv_image, hsv_image, mask=mask)

                v0 = hsv_image_masked[:, :, 0]
                v1 = hsv_image_masked[:, :, 1]
                v2 = hsv_image_masked[:, :, 2]

                plt.subplot(1, 7, 5)
                plt.imshow(v0, cmap='gray')
                plt.title("H Channel")
                plt.axis('off')

                plt.subplot(1, 7, 6)
                plt.imshow(v1, cmap='gray')
                plt.title("S Channel")
                plt.axis('off')

                plt.subplot(1, 7, 7)
                plt.imshow(v2, cmap='gray')
                plt.title("V Channel")
                plt.axis('off')

            plt.show()

directory_path = "D:\Meus Arquivos\ESTUDO\ELT365\PDI\Teste\madeiras_ideais"
process_images_in_directory(directory_path)


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import cv2

def draw_circle_closest_to_center(image, circle_coordinates):
    """
    Draw the largest circle among the three closest to the center of the image.
    
    Args:
        image (numpy.ndarray): The original image.
        circle_coordinates (list of tuples): List of (x, y, r) coordinates of circles.
        
    Returns:
        numpy.ndarray: Image with the circle drawn.
    """
    center_x = image.shape[1] // 2
    center_y = image.shape[0] // 2
    
    circle_coordinates = sorted(circle_coordinates, key=lambda c: np.sqrt((c[0] - center_x)**2 + (c[1] - center_y)**2))
    closest_three_circles = circle_coordinates[:3]
    largest_circle = max(closest_three_circles, key=lambda c: c[2])
    
    if largest_circle is not None:
        (x, y, r) = largest_circle
        cv2.circle(image, (x, y), r, (0, 255, 0), 4)
        cv2.rectangle(image, (x - 5, y - 5), (x + 5, y + 5), (0, 128, 255), -1)
    
    return image, largest_circle

def mask_outside_circle(image, circle):
    """
    Mask the area outside the given circle and draw the circle on the image.
    
    Args:
        image (numpy.ndarray): The input image.
        circle (tuple): The (x, y, r) coordinates of the circle.
        
    Returns:
        numpy.ndarray: The masked image with the circle drawn.
    """
    mask = np.zeros(image.shape[:2], dtype="uint8")
    (x, y, r) = circle
    cv2.circle(mask, (x, y), r, 255, -1)
    
    masked_image = cv2.bitwise_and(image, image, mask=mask)
    cv2.circle(masked_image, (x, y), r, (0, 255, 0), 4)
    
    return masked_image, mask

def imhist(channel, mask, bins=256):
    """Calcula o histograma de um canal de imagem usando uma máscara"""
    hist = cv2.calcHist([channel], [0], mask, [bins], [0, bins])
    return hist.flatten()

def process_images_in_directory(directory_path):
    for filename in os.listdir(directory_path):
        if filename.endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tiff')):
            image_path = os.path.join(directory_path, filename)
            print(f"Processando {image_path}")
            
            image = cv2.imread(image_path)
            image1 = image.copy()
            output = image.copy()
            gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
            edges = cv2.Canny(gray, 100, 250)
            edges = cv2.GaussianBlur(edges, (25, 25), 2)
            edges2 = edges.copy()

            circles_original = cv2.HoughCircles(gray, cv2.HOUGH_GRADIENT, 1.2, 250, minRadius=90, maxRadius=220)
            circles_edges = cv2.HoughCircles(edges, cv2.HOUGH_GRADIENT, 1.2, 200, minRadius=90, maxRadius=300)

            if circles_original is not None:
                circles_original = np.round(circles_original[0, :]).astype("int")
            if circles_edges is not None:
                circles_edges = np.round(circles_edges[0, :]).astype("int")

            if circles_original is not None:
                for (x, y, r) in circles_original:
                    cv2.circle(output, (x, y), r, (0, 255, 0), 4)
                    cv2.rectangle(output, (x - 5, y - 5), (x + 5, y + 5), (0, 128, 255), -1)

            image_rgb = cv2.cvtColor(edges2, cv2.COLOR_BGR2RGB)

            if circles_edges is not None:
                for (x, y, r) in circles_edges:
                    cv2.circle(image_rgb, (x, y), r, (0, 255, 0), 4)
                    cv2.rectangle(image_rgb, (x - 5, y - 5), (x + 5, y + 5), (0, 128, 255), -1)
                    print("Circle detected at center (x={}, y={}), radius={}".format(x, y, r))

            if circles_edges is not None:
                print("Number of circles detected in edges image:", len(circles_edges))

            output_rgb = cv2.cvtColor(output, cv2.COLOR_BGR2RGB)
            edges_rgb = cv2.cvtColor(edges, cv2.COLOR_BGR2RGB)

            plt.figure(figsize=(20, 10))

            plt.subplot(1, 5, 1)
            plt.imshow(image_rgb)
            plt.title("Edges Circles")
            plt.axis('off')

            plt.subplot(1, 5, 2)
            plt.imshow(output_rgb)
            plt.title("Original Circles")
            plt.axis('off')

            if circles_edges is not None:
                closest_circle_edges_output, largest_circle_edges = draw_circle_closest_to_center(image.copy(), circles_edges.tolist())
                
                plt.subplot(1, 5, 3)
                plt.imshow(closest_circle_edges_output)
                plt.title("Closest Circle Edges")
                plt.axis('off')

                masked_edges, mask = mask_outside_circle(image.copy(), largest_circle_edges)
                masked_edges_rgb = cv2.cvtColor(masked_edges, cv2.COLOR_BGR2RGB)
                plt.subplot(1, 5, 4)
                plt.imshow(masked_edges_rgb)
                plt.title("Masked Closest Circle Edges")
                plt.axis('off')

                hsv_image = cv2.cvtColor(masked_edges_rgb, cv2.COLOR_RGB2HSV)
                hsv_image_masked = cv2.bitwise_and(hsv_image, hsv_image, mask=mask)

                v1 = hsv_image_masked[:, :, 1]

                plt.subplot(1, 5, 5)
                plt.imshow(v1, cmap='gray')
                plt.title("S Channel")
                plt.axis('off')

                s_hist = imhist(v1, mask)

                plt.figure()
                plt.plot(np.arange(256), s_hist, 'b', label='Histogram')
                plt.ylabel('Number of Occurrences')
                plt.xlabel('Saturation Level')
                plt.grid(which="both")
                plt.legend()
                plt.show()

            plt.show()

directory_path = "D:\Meus Arquivos\ESTUDO\ELT365\PDI\Teste\madeiras_ideais"
process_images_in_directory(directory_path)


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import cv2

def draw_circle_closest_to_center(image, circle_coordinates):
    """
    Draw the largest circle among the three closest to the center of the image.
    
    Args:
        image (numpy.ndarray): The original image.
        circle_coordinates (list of tuples): List of (x, y, r) coordinates of circles.
        
    Returns:
        numpy.ndarray: Image with the circle drawn.
    """
    center_x = image.shape[1] // 2
    center_y = image.shape[0] // 2
    
    circle_coordinates = sorted(circle_coordinates, key=lambda c: np.sqrt((c[0] - center_x)**2 + (c[1] - center_y)**2))
    closest_three_circles = circle_coordinates[:3]
    largest_circle = max(closest_three_circles, key=lambda c: c[2])
    
    if largest_circle is not None:
        (x, y, r) = largest_circle
        cv2.circle(image, (x, y), r, (0, 255, 0), 4)
        cv2.rectangle(image, (x - 5, y - 5), (x + 5, y + 5), (0, 128, 255), -1)
    
    return image, largest_circle

def mask_outside_circle(image, circle):
    """
    Mask the area outside the given circle and draw the circle on the image.
    
    Args:
        image (numpy.ndarray): The input image.
        circle (tuple): The (x, y, r) coordinates of the circle.
        
    Returns:
        numpy.ndarray: The masked image with the circle drawn.
    """
    mask = np.zeros(image.shape[:2], dtype="uint8")
    (x, y, r) = circle
    cv2.circle(mask, (x, y), r, 255, -1)
    
    masked_image = cv2.bitwise_and(image, image, mask=mask)
    cv2.circle(masked_image, (x, y), r, (0, 255, 0), 4)
    
    return masked_image, mask

def imhist(channel, mask, bins=256):
    """Calcula o histograma de um canal de imagem usando uma máscara"""
    hist = cv2.calcHist([channel], [0], mask, [bins], [0, bins])
    return hist.flatten()

def process_images_in_directory(directory_path):
    for filename in os.listdir(directory_path):
        if filename.endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tiff')):
            image_path = os.path.join(directory_path, filename)
            print(f"Processando {image_path}")
            
            image = cv2.imread(image_path)
            image1 = image.copy()
            output = image.copy()
            gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
            edges = cv2.Canny(gray, 100, 250)
            edges = cv2.GaussianBlur(edges, (25, 25), 2)
            edges2 = edges.copy()

            circles_original = cv2.HoughCircles(gray, cv2.HOUGH_GRADIENT, 1.2, 250, minRadius=90, maxRadius=220)
            circles_edges = cv2.HoughCircles(edges, cv2.HOUGH_GRADIENT, 1.2, 200, minRadius=90, maxRadius=300)

            if circles_original is not None:
                circles_original = np.round(circles_original[0, :]).astype("int")
            if circles_edges is not None:
                circles_edges = np.round(circles_edges[0, :]).astype("int")

            if circles_original is not None:
                for (x, y, r) in circles_original:
                    cv2.circle(output, (x, y), r, (0, 255, 0), 4)
                    cv2.rectangle(output, (x - 5, y - 5), (x + 5, y + 5), (0, 128, 255), -1)

            image_rgb = cv2.cvtColor(edges2, cv2.COLOR_BGR2RGB)

            if circles_edges is not None:
                for (x, y, r) in circles_edges:
                    cv2.circle(image_rgb, (x, y), r, (0, 255, 0), 4)
                    cv2.rectangle(image_rgb, (x - 5, y - 5), (x + 5, y + 5), (0, 128, 255), -1)
                    print("Circle detected at center (x={}, y={}), radius={}".format(x, y, r))

            if circles_edges is not None:
                print("Number of circles detected in edges image:", len(circles_edges))

            output_rgb = cv2.cvtColor(output, cv2.COLOR_BGR2RGB)
            edges_rgb = cv2.cvtColor(edges, cv2.COLOR_BGR2RGB)

            plt.figure(figsize=(20, 10))

            plt.subplot(1, 6, 1)
            plt.imshow(image_rgb)
            plt.title("Edges Circles")
            plt.axis('off')

            plt.subplot(1, 6, 2)
            plt.imshow(output_rgb)
            plt.title("Original Circles")
            plt.axis('off')

            if circles_edges is not None:
                closest_circle_edges_output, largest_circle_edges = draw_circle_closest_to_center(image.copy(), circles_edges.tolist())
                
                plt.subplot(1, 6, 3)
                plt.imshow(closest_circle_edges_output)
                plt.title("Closest Circle Edges")
                plt.axis('off')

                masked_edges, mask = mask_outside_circle(image.copy(), largest_circle_edges)
                masked_edges_rgb = cv2.cvtColor(masked_edges, cv2.COLOR_BGR2RGB)
                plt.subplot(1, 6, 4)
                plt.imshow(masked_edges_rgb)
                plt.title("Masked Closest Circle Edges")
                plt.axis('off')

                hsv_image = cv2.cvtColor(masked_edges_rgb, cv2.COLOR_RGB2HSV)
                hsv_image_masked = cv2.bitwise_and(hsv_image, hsv_image, mask=mask)

                s_channel = hsv_image_masked[:, :, 1]

                plt.subplot(1, 6, 5)
                plt.imshow(s_channel, cmap='gray')
                plt.title("S Channel")
                plt.axis('off')

                s_hist = imhist(s_channel, mask)

                plt.figure()
                plt.plot(np.arange(256), s_hist, 'b', label='Histogram')
                plt.ylabel('Number of Occurrences')
                plt.xlabel('Saturation Level')
                plt.grid(which="both")
                plt.legend()
                plt.show()

                # Create a mask for pixels with saturation between 100 and 200
                s_mask = cv2.inRange(s_channel, 75, 200)

                # Combine the masks to segment only the region of interest within the circle
                combined_mask = cv2.bitwise_and(mask, s_mask)

                # Create the segmented image
                segmented_image = cv2.bitwise_and(masked_edges_rgb, masked_edges_rgb, mask=combined_mask)

                plt.figure()
                plt.imshow(segmented_image)
                plt.title("Segmented Image")
                plt.axis('off')
                plt.show()

            plt.show()

directory_path = "D:\Meus Arquivos\ESTUDO\ELT365\PDI\Teste\madeiras_ideais"
process_images_in_directory(directory_path)


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import cv2

def draw_circle_closest_to_center(image, circle_coordinates):
    """
    Draw the largest circle among the three closest to the center of the image.
    
    Args:
        image (numpy.ndarray): The original image.
        circle_coordinates (list of tuples): List of (x, y, r) coordinates of circles.
        
    Returns:
        numpy.ndarray: Image with the circle drawn.
    """
    center_x = image.shape[1] // 2
    center_y = image.shape[0] // 2
    
    circle_coordinates = sorted(circle_coordinates, key=lambda c: np.sqrt((c[0] - center_x)**2 + (c[1] - center_y)**2))
    closest_three_circles = circle_coordinates[:3]
    largest_circle = max(closest_three_circles, key=lambda c: c[2])
    
    if largest_circle is not None:
        (x, y, r) = largest_circle
        cv2.circle(image, (x, y), r, (0, 255, 0), 4)
        cv2.rectangle(image, (x - 5, y - 5), (x + 5, y + 5), (0, 128, 255), -1)
    
    return image, largest_circle

def mask_outside_circle(image, circle):
    """
    Mask the area outside the given circle and draw the circle on the image.
    
    Args:
        image (numpy.ndarray): The input image.
        circle (tuple): The (x, y, r) coordinates of the circle.
        
    Returns:
        numpy.ndarray: The masked image with the circle drawn.
    """
    mask = np.zeros(image.shape[:2], dtype="uint8")
    (x, y, r) = circle
    cv2.circle(mask, (x, y), r, 255, -1)
    
    masked_image = cv2.bitwise_and(image, image, mask=mask)
    cv2.circle(masked_image, (x, y), r, (0, 255, 0), 4)
    
    return masked_image, mask

def imhist(channel, mask, bins=256):
    """Calcula o histograma de um canal de imagem usando uma máscara"""
    hist = cv2.calcHist([channel], [0], mask, [bins], [0, bins])
    return hist.flatten()

def process_images_in_directory(directory_path):
    for filename in os.listdir(directory_path):
        if filename.endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tiff')):
            image_path = os.path.join(directory_path, filename)
            print(f"Processando {image_path}")
            
            image = cv2.imread(image_path)
            image1 = image.copy()
            output = image.copy()
            gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
            edges = cv2.Canny(gray, 100, 250)
            edges = cv2.GaussianBlur(edges, (25, 25), 2)
            edges2 = edges.copy()

            circles_original = cv2.HoughCircles(gray, cv2.HOUGH_GRADIENT, 1.2, 250, minRadius=90, maxRadius=220)
            circles_edges = cv2.HoughCircles(edges, cv2.HOUGH_GRADIENT, 1.2, 200, minRadius=90, maxRadius=300)

            if circles_original is not None:
                circles_original = np.round(circles_original[0, :]).astype("int")
            if circles_edges is not None:
                circles_edges = np.round(circles_edges[0, :]).astype("int")

            if circles_original is not None:
                for (x, y, r) in circles_original:
                    cv2.circle(output, (x, y), r, (0, 255, 0), 4)
                    cv2.rectangle(output, (x - 5, y - 5), (x + 5, y + 5), (0, 128, 255), -1)

            image_rgb = cv2.cvtColor(edges2, cv2.COLOR_BGR2RGB)

            if circles_edges is not None:
                for (x, y, r) in circles_edges:
                    cv2.circle(image_rgb, (x, y), r, (0, 255, 0), 4)
                    cv2.rectangle(image_rgb, (x - 5, y - 5), (x + 5, y + 5), (0, 128, 255), -1)
                    print("Circle detected at center (x={}, y={}), radius={}".format(x, y, r))

            if circles_edges is not None:
                print("Number of circles detected in edges image:", len(circles_edges))

            output_rgb = cv2.cvtColor(output, cv2.COLOR_BGR2RGB)
            edges_rgb = cv2.cvtColor(edges, cv2.COLOR_BGR2RGB)

            plt.figure(figsize=(20, 10))

            plt.subplot(1, 7, 1)
            plt.imshow(image_rgb)
            plt.title("Edges Circles")
            plt.axis('off')

            plt.subplot(1, 7, 2)
            plt.imshow(output_rgb)
            plt.title("Original Circles")
            plt.axis('off')

            if circles_edges is not None:
                closest_circle_edges_output, largest_circle_edges = draw_circle_closest_to_center(image.copy(), circles_edges.tolist())
                
                plt.subplot(1, 7, 3)
                plt.imshow(closest_circle_edges_output)
                plt.title("Closest Circle Edges")
                plt.axis('off')

                masked_edges, mask = mask_outside_circle(image.copy(), largest_circle_edges)
                masked_edges_rgb = cv2.cvtColor(masked_edges, cv2.COLOR_BGR2RGB)
                plt.subplot(1, 7, 4)
                plt.imshow(masked_edges_rgb)
                plt.title("Masked Closest Circle Edges")
                plt.axis('off')

                hsv_image = cv2.cvtColor(masked_edges_rgb, cv2.COLOR_RGB2HSV)
                hsv_image_masked = cv2.bitwise_and(hsv_image, hsv_image, mask=mask)

                s_channel = hsv_image_masked[:, :, 1]

                plt.subplot(1, 7, 5)
                plt.imshow(s_channel, cmap='gray')
                plt.title("S Channel")
                plt.axis('off')

                s_hist = imhist(s_channel, mask)

                plt.figure()
                plt.plot(np.arange(256), s_hist, 'b', label='Histogram')
                plt.ylabel('Number of Occurrences')
                plt.xlabel('Saturation Level')
                plt.grid(which="both")
                plt.legend()
                plt.show()

                # Create a mask for pixels with saturation between 100 and 200
                s_mask = cv2.inRange(s_channel, 80, 200)

                # Combine the masks to segment only the region of interest within the circle
                combined_mask = cv2.bitwise_and(mask, s_mask)

                # Create the segmented image
                segmented_image = cv2.bitwise_and(masked_edges_rgb, masked_edges_rgb, mask=combined_mask)
                
                plt.figure()
                plt.imshow(segmented_image)
                plt.title("Segmented Image")
                plt.axis('off')
                plt.show()

                # Find contours of the segmented area
                contours, _ = cv2.findContours(combined_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                if contours:
                    # Find the largest contour by area
                    largest_contour = max(contours, key=cv2.contourArea)
                    # Find the minimum enclosing circle for the largest contour
                    (x, y), radius = cv2.minEnclosingCircle(largest_contour)
                    center = (int(x), int(y))
                    radius = int(radius)

                    # Draw the enclosing circle
                    enclosing_circle_image = segmented_image.copy()
                    cv2.circle(enclosing_circle_image, center, radius, (255, 0, 0), 4)  # Draw enclosing circle in red

                    plt.figure()
                    plt.imshow(enclosing_circle_image)
                    plt.title("Enclosing Circle Around Largest Segmented Area")
                    plt.axis('off')
                    plt.show()



directory_path = "D:\Meus Arquivos\ESTUDO\ELT365\PDI\Teste\madeiras_ideais"
process_images_in_directory(directory_path)


In [ ]:
import cv2
import os
import numpy as np
import matplotlib.pyplot as plt

def mask_white_objects(image):
    """
    Cria uma máscara para manter apenas os objetos brancos na imagem e binariza a imagem resultante.

    Args:
        image (numpy.ndarray): A imagem original em BGR.

    Returns:
        numpy.ndarray: A imagem binarizada com apenas os objetos brancos.
    """
    # Converter a imagem para o espaço de cores HSV
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # Definir o intervalo de cores para a cor branca
    lower_white = 210
    upper_white = 250

    # Criar a máscara para manter apenas os objetos brancos
    mask = cv2.inRange(gray, lower_white, upper_white)

    # Aplicar a máscara para obter a imagem binarizada
    result = cv2.bitwise_and(image, image, mask=mask)


    # Binarizar a imagem resultante
    _, binary_image = cv2.threshold(result, 1, 255, cv2.THRESH_BINARY)

    return binary_image

def apply_low_pass_filter(image):
    """
    Aplica um filtro de passa baixa (filtro gaussiano) para suavizar a imagem e reduzir ruídos.

    Args:
        image (numpy.ndarray): A imagem binarizada.

    Returns:
        numpy.ndarray: A imagem suavizada.
    """
    # Aplicar o filtro gaussiano para suavizar a imagem
    blurred_image = cv2.medianBlur(image, 9)
    return blurred_image

def detect_horizontal_lines(image):
    """
    Detecta linhas horizontais em uma imagem binarizada usando a Transformada de Hough.

    Args:
        image (numpy.ndarray): A imagem binarizada.

    Returns:
        list: Lista de linhas detectadas, onde cada linha é representada por (x1, y1, x2, y2).
    """
    # Detectar bordas usando Canny
    edges = cv2.Canny(image, 10, 200, apertureSize=3)

    # Criar um kernel horizontal para a operação morfológica
    horizontal_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (25, 1))
    detected_lines = cv2.morphologyEx(edges, cv2.MORPH_CLOSE, horizontal_kernel)

    # Detectar linhas usando a Transformada de Hough
    lines = cv2.HoughLinesP(detected_lines, 1, np.pi/180, threshold=100, minLineLength=500, maxLineGap=200)
    
    # Filtrar linhas com base no comprimento máximo
    filtered_lines = []
    if lines is not None:
        for line in lines:
            x1, y1, x2, y2 = line[0]
            length = np.sqrt((x2 - x1) ** 2 + (y2 - y1) ** 2)
            if length <= 600:
                filtered_lines.append(line)

    return filtered_lines




    # return lines

def process_images_in_directory(directory_path):
    """
    Processa todas as imagens JPG no diretório especificado, aplicando a máscara para manter apenas os objetos brancos,
    suavizando a imagem binarizada com um filtro de passa baixa e detectando linhas horizontais nas imagens resultantes.

    Args:
        directory_path (str): O caminho para o diretório contendo as imagens.
    """
    for filename in os.listdir(directory_path):
        if filename.lower().endswith('.jpg'):
            image_path = os.path.join(directory_path, filename)
            print(f"Processando {image_path}")

            # Carregar a imagem
            image = cv2.imread(image_path)
            
            if image is None:
                print(f"Error: Não foi possível ler a imagem '{image_path}'. Pulando...")
                continue

            # Aplicar a máscara para manter apenas os objetos brancos
            binary_image = mask_white_objects(image)

            # Aplicar o filtro de passa baixa para suavizar a imagem binarizada
            smoothed_image = apply_low_pass_filter(binary_image)

            # Detectar linhas horizontais na imagem suavizada
            lines = detect_horizontal_lines(smoothed_image)

            A = image.copy()
            

            # Desenhar as linhas detectadas na imagem original
            if lines is not None:
                for line in lines:
                    x1, y1, x2, y2 = line[0]
                    cv2.line(A, (x1, y1), (x2, y2), (0, 255, 0), 2)
            
       

            # Exibir a imagem original com as linhas detectadas e a imagem binarizada suavizada
            plt.figure(figsize=(20, 20))

            plt.subplot(1, 3, 1)
            plt.imshow(cv2.cvtColor(A, cv2.COLOR_BGR2RGB))
            plt.title('Linhas Horizontais Detectadas')
            plt.axis('off')

            plt.subplot(1, 3, 2)
            plt.imshow(binary_image, cmap='gray')
            plt.title('Imagem Binarizada com Objetos Brancos')
            plt.axis('off')

            plt.subplot(1, 3, 3)
            plt.imshow(smoothed_image, cmap='gray')
            plt.title('Imagem Suavizada')
            plt.axis('off')

            plt.show()

# Caminho para o diretório contendo as imagens JPG
directory_path = "D:\Meus Arquivos\ESTUDO\ELT365\PDI\Teste\madeiras_ideais"
process_images_in_directory(directory_path)




In [ ]:
import cv2
import os
import numpy as np
import matplotlib.pyplot as plt

def mask_white_objects(image):
    """
    Cria uma máscara para manter apenas os objetos brancos na imagem e binariza a imagem resultante.

    Args:
        image (numpy.ndarray): A imagem original em BGR.

    Returns:
        numpy.ndarray: A imagem binarizada com apenas os objetos brancos.
    """
    # Converter a imagem para escala de cinza
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # Definir o intervalo de cores para a cor branca
    lower_white = 210
    upper_white = 250

    # Criar a máscara para manter apenas os objetos brancos
    mask = cv2.inRange(gray, lower_white, upper_white)

    # Aplicar a máscara para obter a imagem binarizada
    result = cv2.bitwise_and(image, image, mask=mask)

    # Binarizar a imagem resultante
    _, binary_image = cv2.threshold(result, 1, 255, cv2.THRESH_BINARY)

    return binary_image

def apply_low_pass_filter(image):
    """
    Aplica um filtro de passa baixa (filtro gaussiano) para suavizar a imagem e reduzir ruídos.

    Args:
        image (numpy.ndarray): A imagem binarizada.

    Returns:
        numpy.ndarray: A imagem suavizada.
    """
    # Aplicar o filtro gaussiano para suavizar a imagem
    blurred_image = cv2.medianBlur(image, 9)
    return blurred_image

def apply_high_pass_filter(image):
    """
    Aplica um filtro de passa alta para realçar as bordas na imagem.

    Args:
        image (numpy.ndarray): A imagem suavizada.

    Returns:
        numpy.ndarray: A imagem com realce de bordas.
    """
    # Definir um kernel de nitidez (filtro passa alta)
    kernel = np.array([[-1, -1, -1],
                       [-1,  8, -1],
                       [-1, -1, -1]])

    # Aplicar o kernel de nitidez
    high_pass_image = cv2.filter2D(image, -1, kernel)
    return high_pass_image

def detect_horizontal_lines(image, max_line_length):
    """
    Detecta a maior linha horizontal em uma imagem binarizada usando a Transformada de Hough.

    Args:
        image (numpy.ndarray): A imagem binarizada.
        max_line_length (int): O comprimento máximo permitido para as linhas detectadas.

    Returns:
        tuple: A maior linha detectada representada por (x1, y1, x2, y2).
    """
    # Detectar bordas usando Canny
    edges = cv2.Canny(image, 50, 255, apertureSize=5, L2gradient=True)

    # # Criar um kernel horizontal para a operação morfológica
    # horizontal_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (25, 1))
    # detected_lines = cv2.morphologyEx(edges, cv2.MORPH_CLOSE, horizontal_kernel)

    # Detectar linhas usando a Transformada de Hough
    lines = cv2.HoughLinesP(edges, 1, np.pi/180, threshold=70, minLineLength=500, maxLineGap=90)
    
    # Encontrar a maior linha
    max_length = 0
    longest_line = None
    if lines is not None:
        for line in lines:
            x1, y1, x2, y2 = line[0]
            length = np.sqrt((x2 - x1) ** 2 + (y2 - y1) ** 2)
            if length > max_length and length <= max_line_length:
                max_length = length
                longest_line = (x1, y1, x2, y2)
    
    # Tranformando pixels para milimetros e calculando a proporção
    
    if max_length != 0:
        proporcao = max_length/200
        P = 1 #quantidade de pixels
        comprimnento_milimetro = format((P/proporcao), '.3f')
        max_len = format(max_length, '.3f')

        print(f'Comprimento da linha mais longa: {max_len} pixels')
        print(proporcao)
        print(comprimnento_milimetro)

    else:
        print('Linha não detectada')    

    return longest_line

def process_images_in_directory(directory_path, max_line_length):
    """
    Processa todas as imagens JPG no diretório especificado, aplicando a máscara para manter apenas os objetos brancos,
    suavizando a imagem binarizada com um filtro de passa baixa, aplicando um filtro de passa alta e detectando linhas horizontais nas imagens resultantes.

    Args:
        directory_path (str): O caminho para o diretório contendo as imagens.
        max_line_length (int): O comprimento máximo permitido para as linhas detectadas.
    """
    for filename in os.listdir(directory_path):
        if filename.lower().endswith('.jpg'):
            image_path = os.path.join(directory_path, filename)
            print(f"Processando {image_path}")

            # Carregar a imagem
            image = cv2.imread(image_path)
            
            if image is None:
                print(f"Error: Não foi possível ler a imagem '{image_path}'. Pulando...")
                continue

            # Aplicar a máscara para manter apenas os objetos brancos
            binary_image = mask_white_objects(image)

            # Aplicar o filtro de passa baixa para suavizar a imagem binarizada
            smoothed_image = apply_low_pass_filter(binary_image)

            # Aplicar o filtro de passa alta para realçar as bordas
            high_pass_image = apply_high_pass_filter(smoothed_image)

            # Detectar a maior linha horizontal na imagem suavizada
            longest_line1 = detect_horizontal_lines(smoothed_image, max_line_length)

            # Detectar a maior linha horizontal na imagem com filtro passa alta
            longest_line2 = detect_horizontal_lines(high_pass_image, max_line_length)

            # Desenhar as maiores linhas detectadas na imagem original
            A = image.copy()
            B = image.copy()

            if longest_line1 is not None:
                x1, y1, x2, y2 = longest_line1
                cv2.line(A, (x1, y1), (x2, y2), (0, 255, 0), 2)
        
                # Adicionar comprimento da linha acima da linha
                comprimento_texto = f'Comprimento: 200 mm'
                cv2.putText(A, comprimento_texto, (int((x1+x2)/2), int((y1+y2)/2) - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)

            if longest_line2 is not None:
                x1, y1, x2, y2 = longest_line2
                cv2.line(B, (x1, y1), (x2, y2), (0, 255, 0), 2)

                # Adicionar comprimento da linha acima da linha
                comprimento_texto = f'Comprimento: 200 mm'
                cv2.putText(B, comprimento_texto, (int((x1+x2)/2), int((y1+y2)/2) - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)

            # Exibir a imagem original com as linhas detectadas e a imagem binarizada suavizada
            plt.figure(figsize=(20, 5))

            plt.subplot(1, 5, 1)
            plt.imshow(cv2.cvtColor(A, cv2.COLOR_BGR2RGB))
            plt.title('Maior Linha Horizontal Detectada (Suavizada)')
            plt.axis('off')

            plt.subplot(1, 5, 2)
            plt.imshow(cv2.cvtColor(B, cv2.COLOR_BGR2RGB))
            plt.title('Maior Linha Horizontal Detectada (Passa Alta)')
            plt.axis('off')

            plt.subplot(1, 5, 3)
            plt.imshow(binary_image, cmap='gray')
            plt.title('Imagem Binarizada com Objetos Brancos')
            plt.axis('off')

            plt.subplot(1, 5, 4)
            plt.imshow(smoothed_image, cmap='gray')
            plt.title('Imagem Suavizada')
            plt.axis('off')

            plt.subplot(1, 5, 5)
            plt.imshow(high_pass_image, cmap='gray')
            plt.title('Imagem com Filtro Passa Alta')
            plt.axis('off')

            plt.show()

# Caminho para o diretório contendo as imagens JPG
directory_path = "D:\Meus Arquivos\ESTUDO\ELT365\PDI\Teste\madeiras_ideais"
max_line_length = 750  # Defina o comprimento máximo desejado para as linhas
process_images_in_directory(directory_path, max_line_length)



In [ ]:
#CRIAÇÂO DA LINHA DE REFERÊNCIA:

import cv2
import os
import numpy as np
import matplotlib.pyplot as plt

def mask_white_objects(image):
    """
    Cria uma máscara para manter apenas os objetos brancos na imagem e binariza a imagem resultante.

    Args:
        image (numpy.ndarray): A imagem original em BGR.

    Returns:
        numpy.ndarray: A imagem binarizada com apenas os objetos brancos.
    """
    # Converter a imagem para escala de cinza
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # Definir o intervalo de cores para a cor branca
    lower_white = 210
    upper_white = 250

    # Criar a máscara para manter apenas os objetos brancos
    mask = cv2.inRange(gray, lower_white, upper_white)

    # Aplicar a máscara para obter a imagem binarizada
    result = cv2.bitwise_and(image, image, mask=mask)

    # Binarizar a imagem resultante
    _, binary_image = cv2.threshold(result, 1, 255, cv2.THRESH_BINARY)

    return binary_image

def apply_low_pass_filter(image):
    """
    Aplica um filtro de passa baixa (filtro gaussiano) para suavizar a imagem e reduzir ruídos.

    Args:
        image (numpy.ndarray): A imagem binarizada.

    Returns:
        numpy.ndarray: A imagem suavizada.
    """
    # Aplicar o filtro gaussiano para suavizar a imagem
    blurred_image = cv2.medianBlur(image, 9)
    return blurred_image

def apply_high_pass_filter(image):
    """
    Aplica um filtro de passa alta para realçar as bordas na imagem.

    Args:
        image (numpy.ndarray): A imagem suavizada.

    Returns:
        numpy.ndarray: A imagem com realce de bordas.
    """
    # Definir um kernel de nitidez (filtro passa alta)
    kernel = np.array([[-1, -1, -1],
                       [-1,  8, -1],
                       [-1, -1, -1]])

    # Aplicar o kernel de nitidez
    high_pass_image = cv2.filter2D(image, -1, kernel)
    return high_pass_image

def detect_horizontal_lines(image, max_line_length):
    """
    Detecta a maior linha horizontal em uma imagem binarizada usando a Transformada de Hough.

    Args:
        image (numpy.ndarray): A imagem binarizada.
        max_line_length (int): O comprimento máximo permitido para as linhas detectadas.

    Returns:
        tuple: A maior linha detectada representada por (x1, y1, x2, y2).
    """
    # Detectar bordas usando Canny
    edges = cv2.Canny(image, 50, 255, apertureSize=5, L2gradient=True)

    # # Criar um kernel horizontal para a operação morfológica
    # horizontal_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (25, 1))
    # detected_lines = cv2.morphologyEx(edges, cv2.MORPH_CLOSE, horizontal_kernel)

    # Detectar linhas usando a Transformada de Hough
    lines = cv2.HoughLinesP(edges, 1, np.pi/180, threshold=70, minLineLength=500, maxLineGap=90)
    
    # Encontrar a maior linha
    max_length = 0
    longest_line = None
    if lines is not None:
        for line in lines:
            x1, y1, x2, y2 = line[0]
            length = np.sqrt((x2 - x1) ** 2 + (y2 - y1) ** 2)
            if length > max_length and length <= max_line_length:
                max_length = length
                longest_line = (x1, y1, x2, y2)
    
    # Tranformando pixels para milimetros e calculando a proporção
    
    if max_length != 0:
        proporcao = max_length/200
        P = 1 #quantidade de pixels
        comprimnento_milimetro = format((P/proporcao), '.3f')
        max_len = format(max_length, '.3f')

        print(f'Comprimento da linha mais longa: {max_len} pixels')
        print(proporcao)
        print(comprimnento_milimetro)

    else:
        print('Linha não detectada')    

    return longest_line

def process_images_in_directory(directory_path, max_line_length):
    """
    Processa todas as imagens JPG no diretório especificado, aplicando a máscara para manter apenas os objetos brancos,
    suavizando a imagem binarizada com um filtro de passa baixa, aplicando um filtro de passa alta e detectando linhas horizontais nas imagens resultantes.

    Args:
        directory_path (str): O caminho para o diretório contendo as imagens.
        max_line_length (int): O comprimento máximo permitido para as linhas detectadas.
    """
    for filename in os.listdir(directory_path):
        if filename.lower().endswith('.jpg'):
            image_path = os.path.join(directory_path, filename)
            print(f"Processando {image_path}")

            # Carregar a imagem
            image = cv2.imread(image_path)
            
            if image is None:
                print(f"Error: Não foi possível ler a imagem '{image_path}'. Pulando...")
                continue

            # Aplicar a máscara para manter apenas os objetos brancos
            binary_image = mask_white_objects(image)

            # Aplicar o filtro de passa baixa para suavizar a imagem binarizada
            smoothed_image = apply_low_pass_filter(binary_image)

            # Aplicar o filtro de passa alta para realçar as bordas
            high_pass_image = apply_high_pass_filter(smoothed_image)

            # Detectar a maior linha horizontal na imagem com filtro passa alta
            longest_line2 = detect_horizontal_lines(high_pass_image, max_line_length)

            # Desenhar as maiores linhas detectadas na imagem original
            
            B = image.copy()

            if longest_line2 is not None:
                x1, y1, x2, y2 = longest_line2
                cv2.line(B, (x1, y1), (x2, y2), (0, 255, 0), 2)

                # Adicionar comprimento da linha acima da linha
                comprimento_texto = f'Comprimento: 200 mm'
                cv2.putText(B, comprimento_texto, (int((x1+x2)/2), int((y1+y2)/2) - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)

            # Exibir a imagem original com as linhas detectadas e a imagem binarizada suavizada
            plt.figure(figsize=(20, 5))
            plt.imshow(cv2.cvtColor(B, cv2.COLOR_BGR2RGB))
            plt.title('Maior Linha Detectada')
            plt.axis('off')

            plt.show()

# Caminho para o diretório contendo as imagens JPG
directory_path = "D:\Meus Arquivos\ESTUDO\ELT365\PDI\Teste\madeiras_ideais"
max_line_length = 750  # Defina o comprimento máximo desejado para as linhas
process_images_in_directory(directory_path, max_line_length)



In [ ]:
#CRIAÇÃO DO MENOR CIRCULO E RELAÇÃO ENTRE AS ÁREAS:

import os
import numpy as np
import matplotlib.pyplot as plt
import cv2

def draw_circle_closest_to_center(image, circle_coordinates):
    """
    Draw the largest circle among the three closest to the center of the image.
    
    Args:
        image (numpy.ndarray): The original image.
        circle_coordinates (list of tuples): List of (x, y, r) coordinates of circles.
        
    Returns:
        numpy.ndarray: Image with the circle drawn.
    """
    center_x = image.shape[1] // 2
    center_y = image.shape[0] // 2
    
    circle_coordinates = sorted(circle_coordinates, key=lambda c: np.sqrt((c[0] - center_x)**2 + (c[1] - center_y)**2))
    closest_three_circles = circle_coordinates[:3]
    largest_circle = max(closest_three_circles, key=lambda c: c[2])
    
    if largest_circle is not None:
        (x, y, r) = largest_circle
        cv2.circle(image, (x, y), r, (0, 255, 0), 4)
        cv2.rectangle(image, (x - 5, y - 5), (x + 5, y + 5), (0, 128, 255), -1)
    
    return image, largest_circle

def mask_outside_circle(image, circle):
    """
    Mask the area outside the given circle and draw the circle on the image.
    
    Args:
        image (numpy.ndarray): The input image.
        circle (tuple): The (x, y, r) coordinates of the circle.
        
    Returns:
        numpy.ndarray: The masked image with the circle drawn.
    """
    mask = np.zeros(image.shape[:2], dtype="uint8")
    (x, y, r) = circle
    cv2.circle(mask, (x, y), r, 255, -1)
    
    masked_image = cv2.bitwise_and(image, image, mask=mask)
    cv2.circle(masked_image, (x, y), r, (0, 255, 0), 4)
    
    return masked_image, mask

def imhist(channel, mask, bins=256):
    """Calcula o histograma de um canal de imagem usando uma máscara"""
    hist = cv2.calcHist([channel], [0], mask, [bins], [0, bins])
    return hist.flatten()





def mask_white_objects(image):
    """
    Cria uma máscara para manter apenas os objetos brancos na imagem e binariza a imagem resultante.

    Args:
        image (numpy.ndarray): A imagem original em BGR.

    Returns:
        numpy.ndarray: A imagem binarizada com apenas os objetos brancos.
    """
    # Converter a imagem para escala de cinza
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # Definir o intervalo de cores para a cor branca
    lower_white = 210
    upper_white = 250

    # Criar a máscara para manter apenas os objetos brancos
    mask = cv2.inRange(gray, lower_white, upper_white)

    # Aplicar a máscara para obter a imagem binarizada
    result = cv2.bitwise_and(image, image, mask=mask)

    # Binarizar a imagem resultante
    _, binary_image = cv2.threshold(result, 1, 255, cv2.THRESH_BINARY)

    return binary_image

def apply_low_pass_filter(image):
    """
    Aplica um filtro de passa baixa (filtro gaussiano) para suavizar a imagem e reduzir ruídos.

    Args:
        image (numpy.ndarray): A imagem binarizada.

    Returns:
        numpy.ndarray: A imagem suavizada.
    """
    # Aplicar o filtro gaussiano para suavizar a imagem
    blurred_image = cv2.medianBlur(image, 9)
    return blurred_image

def apply_high_pass_filter(image):
    """
    Aplica um filtro de passa alta para realçar as bordas na imagem.

    Args:
        image (numpy.ndarray): A imagem suavizada.

    Returns:
        numpy.ndarray: A imagem com realce de bordas.
    """
    # Definir um kernel de nitidez (filtro passa alta)
    kernel = np.array([[-1, -1, -1],
                       [-1,  8, -1],
                       [-1, -1, -1]])

    # Aplicar o kernel de nitidez
    high_pass_image = cv2.filter2D(image, -1, kernel)
    return high_pass_image

def detect_horizontal_lines(image, max_line_length):
    """
    Detecta a maior linha horizontal em uma imagem binarizada usando a Transformada de Hough.

    Args:
        image (numpy.ndarray): A imagem binarizada.
        max_line_length (int): O comprimento máximo permitido para as linhas detectadas.

    Returns:
        tuple: A maior linha detectada representada por (x1, y1, x2, y2).
    """
    # Detectar bordas usando Canny
    edges = cv2.Canny(image, 50, 255, apertureSize=5, L2gradient=True)

    # # Criar um kernel horizontal para a operação morfológica
    # horizontal_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (25, 1))
    # detected_lines = cv2.morphologyEx(edges, cv2.MORPH_CLOSE, horizontal_kernel)

    # Detectar linhas usando a Transformada de Hough
    lines = cv2.HoughLinesP(edges, 1, np.pi/180, threshold=70, minLineLength=500, maxLineGap=90)
    
    # Encontrar a maior linha
    max_length = 0
    longest_line = None
    if lines is not None:
        for line in lines:
            x1, y1, x2, y2 = line[0]
            length = np.sqrt((x2 - x1) ** 2 + (y2 - y1) ** 2)
            if length > max_length and length <= max_line_length:
                max_length = length
                longest_line = (x1, y1, x2, y2)
    
    # Tranformando pixels para milimetros e calculando a proporção
    
    if max_length != 0:
        proporcao = max_length/200
        
        return longest_line, proporcao

    else:
        print('Linha não detectada')    

        return None, None







def process_images_in_directory(directory_path,max_line_length):

    #SEGMENTAÇÃO DO TRONCO:
    for filename in os.listdir(directory_path):
        if filename.endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tiff')):
            image_path = os.path.join(directory_path, filename)
            print(f"Processando {image_path}")
            
            image = cv2.imread(image_path)
            image1 = image.copy()
            output = image.copy()
            gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
            edges = cv2.Canny(gray, 100, 250)
            edges = cv2.GaussianBlur(edges, (25, 25), 2)
            edges2 = edges.copy()

            circles_original = cv2.HoughCircles(gray, cv2.HOUGH_GRADIENT, 1.2, 250, minRadius=90, maxRadius=220)
            circles_edges = cv2.HoughCircles(edges, cv2.HOUGH_GRADIENT, 1.2, 200, minRadius=90, maxRadius=300)

            if circles_original is not None:
                circles_original = np.round(circles_original[0, :]).astype("int")
            if circles_edges is not None:
                circles_edges = np.round(circles_edges[0, :]).astype("int")

            if circles_original is not None:
                for (x, y, r) in circles_original:
                    cv2.circle(output, (x, y), r, (0, 255, 0), 4)
                    cv2.rectangle(output, (x - 5, y - 5), (x + 5, y + 5), (0, 128, 255), -1)

            image_rgb = cv2.cvtColor(edges2, cv2.COLOR_BGR2RGB)

            if circles_edges is not None:
                for (x, y, r) in circles_edges:
                    cv2.circle(image_rgb, (x, y), r, (0, 255, 0), 4)
                    cv2.rectangle(image_rgb, (x - 5, y - 5), (x + 5, y + 5), (0, 128, 255), -1)
                    # print("Circle detected at center (x={}, y={}), radius={}".format(x, y, r))

            if circles_edges is not None:
                print("Number of circles detected in edges image:", len(circles_edges))

            output_rgb = cv2.cvtColor(output, cv2.COLOR_BGR2RGB)
            edges_rgb = cv2.cvtColor(edges, cv2.COLOR_BGR2RGB)

            plt.figure(figsize=(20, 5))

            if circles_edges is not None:
                closest_circle_edges_output, largest_circle_edges = draw_circle_closest_to_center(image.copy(), circles_edges.tolist())

                raio_1 = largest_circle_edges[2]

                

                masked_edges, mask = mask_outside_circle(image.copy(), largest_circle_edges)
                masked_edges_rgb = cv2.cvtColor(masked_edges, cv2.COLOR_BGR2RGB)

                hsv_image = cv2.cvtColor(masked_edges_rgb, cv2.COLOR_RGB2HSV)
                hsv_image_masked = cv2.bitwise_and(hsv_image, hsv_image, mask=mask)

                s_channel = hsv_image_masked[:, :, 1]
                s_hist = imhist(s_channel, mask)

                # Create a mask for pixels with saturation between 100 and 200
                s_mask = cv2.inRange(s_channel, 80, 200)

                # Combine the masks to segment only the region of interest within the circle
                combined_mask = cv2.bitwise_and(mask, s_mask)

                # Create the segmented image
                segmented_image = cv2.bitwise_and(masked_edges_rgb, masked_edges_rgb, mask=combined_mask)
                

                # Find contours of the segmented area
                contours, _ = cv2.findContours(combined_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                if contours:
                    # Find the largest contour by area
                    largest_contour = max(contours, key=cv2.contourArea)
                    # Find the minimum enclosing circle for the largest contour
                    (x, y), radius = cv2.minEnclosingCircle(largest_contour)
                    center = (int(x), int(y))
                    radius = int(radius)

                    

                    cv2.circle(closest_circle_edges_output, center, radius, (255, 0, 0), 4)  # Draw enclosing circle in red
                    plt.subplot(1, 3, 3)
                    plt.imshow(closest_circle_edges_output)
                    plt.title("Enclosing Circle Around Largest Segmented Area")
                    plt.axis('off')
                    plt.show()
            
            #CRIAÇÃO DA LINHA DE REFERÊNCIA:
            """
            Processa todas as imagens JPG no diretório especificado, aplicando a máscara para manter apenas os objetos brancos,
            suavizando a imagem binarizada com um filtro de passa baixa, aplicando um filtro de passa alta e detectando linhas horizontais nas imagens resultantes.

            Args:
                directory_path (str): O caminho para o diretório contendo as imagens.
                max_line_length (int): O comprimento máximo permitido para as linhas detectadas.
            """
                
            if image is None:
                print(f"Error: Não foi possível ler a imagem '{image_path}'. Pulando...")
                continue

            # Aplicar a máscara para manter apenas os objetos brancos
            binary_image = mask_white_objects(image)

            # Aplicar o filtro de passa baixa para suavizar a imagem binarizada
            smoothed_image = apply_low_pass_filter(binary_image)

            # Aplicar o filtro de passa alta para realçar as bordas
            high_pass_image = apply_high_pass_filter(smoothed_image)

            # Detectar a maior linha horizontal na imagem com filtro passa alta
            longest_line2, proporcao = detect_horizontal_lines(high_pass_image, max_line_length)

            # Desenhar as maiores linhas detectadas na imagem original
            
            B = closest_circle_edges_output

            if longest_line2 is not None:
                x1, y1, x2, y2 = longest_line2
                cv2.line(B, (x1, y1), (x2, y2), (0, 255, 0), 2)

                # Adicionar comprimento da linha acima da linha
                comprimento_texto = f'Comprimento: 200 mm'
                cv2.putText(B, comprimento_texto, (int((x1+x2)/2), int((y1+y2)/2) - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)

            # Exibir a imagem original com as linhas detectadas e a imagem binarizada suavizada
            plt.figure(figsize=(20, 5))
            plt.imshow(cv2.cvtColor(B, cv2.COLOR_BGR2RGB))
            plt.title('Resultado final')
            plt.axis('off')

            plt.show()

            print(f'Raio da região escura = {radius} pixels')
            print(f"Raio do tronco em pixels = {raio_1} pixels")

            if proporcao != None:

                diametro_tronco = 2*raio_1
                diametro_Pescura = 2*radius
            
                diametro_milimetro_tronco = diametro_tronco/proporcao
                diametro_milimetro_Pescura = diametro_Pescura/proporcao

                area_tronco = (np.pi)*(diametro_milimetro_tronco/2)**2
                area_Pescura = (np.pi)*(diametro_milimetro_Pescura/2)**2

                prop_area = (area_Pescura/area_tronco)*100

                Z = format(diametro_milimetro_tronco, '.3f')
                X = format(diametro_milimetro_Pescura, '.3f')
                C = format(area_tronco, '.3f')
                V = format(area_Pescura, '.3f')
                N = format(prop_area, '.3f')

                print(f'Diâmetro do tronco em milimetros = {Z} mm')
                print(f'Diâmetro da região mais escura = {X} mm')
                
                print(f'Área do tronco = {C} mm²')
                print(f'Área da região mais escura = {V} mm²')
                
                print(f'Proporção entre as áreas = {N}%')
                print('--------------------')

            else:
                print('Linha de referência não detectada')
                print('--------------------')



directory_path = "D:\Meus Arquivos\ESTUDO\ELT365\PDI\Teste\madeiras_ideais"
max_line_length = 750  # Defina o comprimento máximo desejado para as linhas
process_images_in_directory(directory_path, max_line_length)
